# NMDpredictionmodel: Guided Reproducibility Pipeline

This notebook provides a single entry point for reproducing the annotation,
feature-generation, feature-matrix construction, and TrunCat modeling workflow.

The notebook calls the scripts contained in this repository rather than
reimplementing their analysis logic.

## Supported datasets

- TOPMed
- gnomAD
- ClinVar
- GREGoR

## Reproducibility and TOPMed data access

The original TOPMed WGS and RNA-seq data used for variant and
allele-specific expression extraction are controlled-access and cannot be
distributed through this repository.

The `Dataset_extraction/` directory documents the preprocessing workflow used
to generate the analysis input from authorized TOPMed data.

For reproducibility, the default TOPMed workflow in this notebook begins from
the de-identified variant-level analysis input provided with the repository.
From this starting point, the notebook runs the annotation, filtering,
feature-generation, feature-matrix construction, and TrunCat modeling workflow.

Users with authorized access to the original TOPMed data can additionally use
the scripts in `Dataset_extraction/` to reproduce the upstream extraction
steps.

The shared annotation and feature-generation modules are also structured for
application to independent variant datasets prepared in the documented input
format.

## Main stages

1. Clone the repository
2. Install software dependencies
3. Configure dataset and reference paths
4. Load the analysis-ready variant input
5. Run variant and transcript annotation
6. Apply shared variant filtering
7. Generate sequence, transcript, gene, and variant-level features
8. Build the core feature table
9. Generate additional TrunCat feature families
10. Assemble the TrunCat feature matrix
11. Perform feature preprocessing and selection
12. Train and evaluate the TrunCat model

In [1]:
REPO_URL = "https://github.com/CobanAkdemirlab/NMDpredictionmodel.git"
PROJECT_DIR = "/content/NMDpredictionmodel"

import os
import subprocess

if not os.path.exists(PROJECT_DIR):
    subprocess.run(
        ["git", "clone", REPO_URL, PROJECT_DIR],
        check=True
    )

os.chdir(PROJECT_DIR)

print("Repository ready:")
print(PROJECT_DIR)

Repository ready:
/content/NMDpredictionmodel


In [2]:
%%bash
set -e

apt-get update -qq

apt-get install -y -qq \
    bcftools \
    tabix \
    samtools \
    bedtools \
    libcurl4-openssl-dev \
    libssl-dev \
    libxml2-dev

(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack .../0-libxml2-dev_2.9.14+dfsg-1.3ubuntu3.9_amd64.deb ...
Unpacking libxml2-dev:amd64 (2.9.14+dfsg-1.3ubuntu3.9) over (2.9.14+dfsg-1.3ubuntu3.8) ...
Preparing to unpack .../1-libxml2_2.9.14+dfsg-1.3ubuntu3.9_amd64.deb ...
Unpacking libxml2:amd64 (2.9.14+dfsg-1.3ubuntu3.9) over (2.9.14+dfsg-1.3ubuntu3.8) ...
Selecting previously unselected package libhtscodecs2:amd64.
Preparing to unpack .../2-libhtscodecs2_1.6.0-1build1_amd64.deb ...
Unpacking libhtscodecs2:amd64 (1.6.0-1build1) ...
Selecting previously unselected package libhts3t64:amd64.
Preparing to unpack .../3-libhts3t64_1.19+ds-1.1build3_amd64.deb ...
Unpacking libhts3t64:amd64 (1.19+ds-1.1build3) ...
Selecting previously unselected package bcftools.
Preparing to unpack .../4-bcftools_1.19-1build2_amd64.deb ...
Unpacking bcftools (1.19-1build2) ...
Selecting previously unselected package bedtools.
Preparing to unpack .../5-bedtools_2.31.1+

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [3]:
%pip install -q \
    pandas \
    numpy \
    pysam \
    pyfaidx \
    gffutils \
    pyranges \
    pyBigWig \
    biopython \
    tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 592.4/592.4 kB 31.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 78.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.1/189.1 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.2 MB/s eta 0:00:00


In [ ]:
%%bash
Rscript - <<'RSCRIPT'

cran_packages <- c(
    "dplyr",
    "tidyr",
    "data.table",
    "readr",
    "readxl",
    "stringr"
)

installed <- rownames(installed.packages())

for (pkg in cran_packages) {
    if (!pkg %in% installed) {
        install.packages(
            pkg,
            repos = "https://cloud.r-project.org"
        )
    }
}

if (!requireNamespace("BiocManager", quietly = TRUE)) {
    install.packages(
        "BiocManager",
        repos = "https://cloud.r-project.org"
    )
}

bioc_packages <- c(
    "GenomicRanges",
    "GenomicFeatures",
    "Biostrings",
    "VariantAnnotation",
    "rtracklayer",
    "AnnotationDbi",
    "biomaRt"
)

for (pkg in bioc_packages) {
    if (!requireNamespace(pkg, quietly = TRUE)) {
        BiocManager::install(
            pkg,
            ask = FALSE,
            update = FALSE
        )
    }
}

RSCRIPT

## External software

The following tools are used by parts of the pipeline:

- ANNOVAR
- GATK
- bcftools
- tabix
- MEME Suite / FIMO 5.5.5

ANNOVAR and some controlled-access datasets cannot be automatically
distributed through this notebook.

Before running the corresponding steps, provide paths to locally available
installations/files in the configuration section below.

In [ ]:
from pathlib import Path

# ------------------------------------------------------------------
# Dataset
# ------------------------------------------------------------------

DATASET = "TOPMed"

ALLOWED_DATASETS = {
    "TOPMed",
    "gnomAD",
    "ClinVar",
    "GREGoR"
}

assert DATASET in ALLOWED_DATASETS


# ------------------------------------------------------------------
# Project
# ------------------------------------------------------------------

PROJECT_DIR = Path("/content/NMDpredictionmodel")


# ------------------------------------------------------------------
# Reference genome / transcript annotation
# ------------------------------------------------------------------

REFERENCE_DIR = Path("/content/reference")

GTF_FILE = (
    REFERENCE_DIR
    / "gencode.v26.primary_assembly.annotation.gtf.gz"
)

HG38_FASTA = (
    REFERENCE_DIR
    / "hg38.fa"
)


# ------------------------------------------------------------------
# GTEx v8
# ------------------------------------------------------------------

GTEX_EXPRESSION = (
    REFERENCE_DIR
    / "GTEx_Analysis_2017-06-05_v8_RNASeQCv1.1.9_gene_median_tpm.gct"
)

GTEX_EGENES = (
    REFERENCE_DIR
    / "Whole_Blood.v8.egenes.txt"
)


# ------------------------------------------------------------------
# Conservation
# ------------------------------------------------------------------

PHASTCONS_BW = (
    REFERENCE_DIR
    / "hg38.phastCons100way.bw"
)

PHYLOP_BW = (
    REFERENCE_DIR
    / "hg38.phyloP100way.bw"
)


# ------------------------------------------------------------------
# EJC occupancy
# ------------------------------------------------------------------

EJC_FILE = (
    REFERENCE_DIR
    / "ejc_occupancy_intervals.bed"
)


# ------------------------------------------------------------------
# Codon optimality
# ------------------------------------------------------------------

TRNA_TABLE = (
    PROJECT_DIR
    / "Features"
    / "reference"
    / "codon_optimality"
    / "hg38_UCSC_tRNA_table.tsv"
)

OPTIMAL_CODONS = (
    PROJECT_DIR
    / "Features"
    / "reference"
    / "codon_optimality"
    / "optimal_codons.txt"
)


# ------------------------------------------------------------------
# External software
# ------------------------------------------------------------------

ANNOVAR_DIR = Path("/content/annovar")

FIMO_EXECUTABLE = "fimo"


# ------------------------------------------------------------------
# Input/output
# ------------------------------------------------------------------

DATASET_OUTPUT = (
    PROJECT_DIR
    / "output"
    / DATASET
)

DATASET_OUTPUT.mkdir(
    parents=True,
    exist_ok=True
)

print("Dataset:", DATASET)
print("Output:", DATASET_OUTPUT)

In [ ]:
def report_file(label, path):
    path = Path(path)

    if path.exists():
        print(f"[OK]      {label}: {path}")
    else:
        print(f"[MISSING] {label}: {path}")


report_file(
    "GENCODE v26 GTF",
    GTF_FILE
)

report_file(
    "hg38 FASTA",
    HG38_FASTA
)

report_file(
    "GTEx expression",
    GTEX_EXPRESSION
)

report_file(
    "GTEx Whole Blood eGenes",
    GTEX_EGENES
)

report_file(
    "phastCons100way",
    PHASTCONS_BW
)

report_file(
    "phyloP100way",
    PHYLOP_BW
)

report_file(
    "EJC occupancy",
    EJC_FILE
)

report_file(
    "optimal codons",
    OPTIMAL_CODONS
)

## Dataset extraction

Run the dataset-specific extraction script before entering the shared
annotation workflow.

For controlled-access datasets such as TOPMed, authorized source files must
already be available.

Select and run the appropriate extraction module below.

In [ ]:
import subprocess

if DATASET == "TOPMed":

    print(
        "TOPMed requires authorized WGS/RNA-seq inputs."
    )

    print(
        "Run Dataset_extraction/01a_TOPMed_preprocessing.sh "
        "and Dataset_extraction/01b_TOPMed_extraction.R "
        "after configuring their input paths."
    )

elif DATASET == "gnomAD":

    subprocess.run(
        [
            "Rscript",
            "Dataset_extraction/02_gnomAD_extraction.R"
        ],
        check=True
    )

elif DATASET == "ClinVar":

    subprocess.run(
        [
            "Rscript",
            "Dataset_extraction/03_ClinVar_extraction.R"
        ],
        check=True
    )

elif DATASET == "GREGoR":

    subprocess.run(
        [
            "Rscript",
            "Dataset_extraction/04_GREGoR_extraction.R"
        ],
        check=True
    )

## Shared annotation

The following steps are shared across TOPMed, gnomAD, ClinVar, and GREGoR:

1. Canonical transcript / ANNOVAR annotation
2. GENCODE v26 transcript structure
3. PTC annotation
4. Shared analysis filtering

In [ ]:
subprocess.run(
    [
        "Rscript",
        "Annotation/01_variant_annotation.R",
        "--dataset",
        DATASET
    ],
    check=True
)

In [ ]:
TRANSCRIPT_STRUCTURE = (
    PROJECT_DIR
    / "output"
    / "reference"
    / "gencode_v26_transcript_structure.rds"
)

if TRANSCRIPT_STRUCTURE.exists():

    print(
        "GENCODE v26 transcript structure already exists."
    )

else:

    subprocess.run(
        [
            "Rscript",
            "Annotation/02_GENCODEv26_transcript_structure.R"
        ],
        check=True
    )

In [ ]:
subprocess.run(
    [
        "Rscript",
        "Annotation/03_PTC_annotation.R",
        "--dataset",
        DATASET
    ],
    check=True
)

In [ ]:
subprocess.run(
    [
        "Rscript",
        "Annotation/04_shared_variant_filtering.R",
        "--dataset",
        DATASET,
        "--gtex-expression",
        str(GTEX_EXPRESSION),
        "--gtex-egenes",
        str(GTEX_EGENES)
    ],
    check=True
)

In [ ]:
if DATASET == "TOPMed":

    subprocess.run(
        [
            "Rscript",
            "Annotation/05_TOPMed_ASE_simulation.R"
        ],
        check=True
    )

else:

    print(
        "Skipping TOPMed ASE simulation for",
        DATASET
    )

# Feature generation

Feature modules 01–07 are assembled into the core feature table.

Modules 08–12 remain separate because the TrunCat modeling workflow merges
those annotation files later.

In [ ]:
feature_scripts = [
    "Features/01_GENCODEv26_sequence_features.R",
    "Features/02_PTBP1_binding_features.R",
    "Features/03_PTC_amino_acid_context.R",
    "Features/04_gene_level_features.R",
    "Features/05_PTC_geometry_derived_features.R",
]

for script in feature_scripts:

    print(
        "\nRunning:",
        script
    )

    subprocess.run(
        [
            "Rscript",
            script,
            "--dataset",
            DATASET
        ],
        check=True
    )

In [ ]:
subprocess.run(
    [
        "Rscript",
        "Features/06a_prepare_variant_scores.R",
        "--dataset",
        DATASET
    ],
    check=True
)

print(
    "Prepared ANNOVAR input."
)

print(
    "Run 06b_run_variant_scores.sh after configuring ANNOVAR."
)

In [ ]:
subprocess.run(
    [
        "bash",
        "Features/06b_run_variant_scores.sh",
        DATASET
    ],
    check=True
)

subprocess.run(
    [
        "Rscript",
        "Features/06c_merge_variant_scores.R",
        "--dataset",
        DATASET
    ],
    check=True
)

In [ ]:
subprocess.run(
    [
        "python",
        "Features/07a_motif_region_extraction.py"
    ],
    check=True
)

In [ ]:
import shutil
import subprocess

fimo_path = shutil.which(
    "fimo"
)

if fimo_path is None:

    print(
        "FIMO is not installed or not available on PATH."
    )

    print(
        "The manuscript analysis used MEME Suite 5.5.5."
    )

else:

    result = subprocess.run(
        [
            "fimo",
            "--version"
        ],
        capture_output=True,
        text=True
    )

    print(
        result.stdout or
        result.stderr
    )

In [ ]:
# Build core feature table
subprocess.run(
    [
        "Rscript",
        "Features/99_build_core_feature_table.R",
        "--dataset",
        DATASET
    ],
    check=True
)

# Expected output
CORE_OUTPUT = (
    PROJECT_DIR
    / "output"
    / DATASET
    / f"{DATASET}_core_features.csv"
)

print("Expected core feature table:")
print(CORE_OUTPUT)

## Codon optimality

The optimal codon reference was derived from the hg38 UCSC Table Browser
GtRNAdb-based tRNA track.

If `optimal_codons.txt` is already present, this step may be skipped.

In [ ]:
if OPTIMAL_CODONS.exists():

    print(
        "optimal_codons.txt already exists."
    )

else:

    subprocess.run(
        [
            "python",
            "Features/08a_make_optimal_codons_from_trna.py",
            "--input",
            str(TRNA_TABLE),
            "--output",
            str(OPTIMAL_CODONS)
        ],
        check=True
    )

In [ ]:
ANNOTATION_DIR = (
    PROJECT_DIR
    / "Model"
    / "TrunCat"
    / "data"
    / "annotations"
)

ANNOTATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CODON_OUTPUT = (
    ANNOTATION_DIR
    / "codon_optimality.tsv"
)

subprocess.run(
    [
        "python",
        "Features/08b_codon_optimality_features.py",
        "--variants",
        str(CORE_OUTPUT),
        "--gtf",
        str(GTF_FILE),
        "--fasta",
        str(HG38_FASTA),
        "--optimal_codons",
        str(OPTIMAL_CODONS),
        "--out",
        str(CODON_OUTPUT),
        "--window_nt",
        "100",
        "--aggregate",
        "none"
    ],
    check=True
)

In [ ]:
EJC_OUTPUT = (
    ANNOTATION_DIR
    / "ejc_occupancy.tsv"
)

subprocess.run(
    [
        "python",
        "Features/10_EJC_occupancy_features.py",
        "--input",
        str(CORE_OUTPUT),
        "--gtf",
        str(GTF_FILE),
        "--ejc",
        str(EJC_FILE),
        "--output",
        str(EJC_OUTPUT)
    ],
    check=True
)

In [ ]:
AUG_OUTPUT = (
    ANNOTATION_DIR
    / "ptc_aug.tsv"
)

AUG_DB = (
    DATASET_OUTPUT
    / "ptc_aug_cache.sqlite"
)

subprocess.run(
    [
        "python",
        "Features/11_PTC_AUG_features.py",
        "--input",
        str(CORE_OUTPUT),
        "--fasta",
        str(HG38_FASTA),
        "--gtf",
        str(GTF_FILE),
        "--db",
        str(AUG_DB),
        "--output",
        str(AUG_OUTPUT),
        "--build-db"
    ],
    check=True
)

In [ ]:
READTHROUGH_OUTPUT = (
    ANNOTATION_DIR
    / "readthrough.csv"
)

subprocess.run(
    [
        "python",
        "Features/12_readthrough_features.py",
        "--input",
        str(CORE_OUTPUT),
        "--gtf",
        str(GTF_FILE),
        "--genome",
        str(HG38_FASTA),
        "--output",
        str(READTHROUGH_OUTPUT)
    ],
    check=True
)

In [ ]:
if DATASET == "TOPMed":

    required_model_inputs = [
        PROJECT_DIR / "Model/TrunCat/data/TOPMed_stopgain.csv",
        ANNOTATION_DIR / "codon_optimality.tsv",
        ANNOTATION_DIR / "readthrough.csv",
        ANNOTATION_DIR / "ejc_occupancy.tsv",
        ANNOTATION_DIR / "ptc_aug.tsv",
        ANNOTATION_DIR / "conservation_medians.csv",
    ]

    print(
        "TrunCat input check:\n"
    )

    all_ready = True

    for path in required_model_inputs:

        if path.exists():

            print(
                "[OK]     ",
                path.name
            )

        else:

            print(
                "[MISSING]",
                path
            )

            all_ready = False

    if all_ready:

        print(
            "\nAll TrunCat model inputs are available."
        )

In [ ]:
if DATASET == "TOPMed":

    subprocess.run(
        [
            "python",
            "Model/TrunCat/scripts/01_data_loading_and_merging.py"
        ],
        check=True
    )

In [ ]:
if DATASET == "TOPMed":

    subprocess.run(
        [
            "python",
            "Model/TrunCat/scripts/02_feature_cleaning_and_selection.py"
        ],
        check=True
    )

In [ ]:
if DATASET == "TOPMed":

    subprocess.run(
        [
            "python",
            "Model/TrunCat/scripts/03_model_training.py"
        ],
        check=True
    )

In [ ]:
from pathlib import Path
import pandas as pd

print("=" * 70)
print("NMDpredictionmodel pipeline summary")
print("=" * 70)

if CORE_OUTPUT.exists():

    core = pd.read_csv(
        CORE_OUTPUT,
        low_memory=False
    )

    print(
        "Core feature table:",
        core.shape
    )

if DATASET == "TOPMed":

    merged = (
        PROJECT_DIR
        / "Model"
        / "TrunCat"
        / "data"
        / "TOPMed_merged.csv"
    )

    cleaned = (
        PROJECT_DIR
        / "Model"
        / "TrunCat"
        / "data"
        / "TOPMed_cleaned.csv"
    )

    if merged.exists():

        merged_df = pd.read_csv(
            merged,
            low_memory=False
        )

        print(
            "Merged TrunCat table:",
            merged_df.shape
        )

    if cleaned.exists():

        cleaned_df = pd.read_csv(
            cleaned,
            low_memory=False
        )

        print(
            "Cleaned TrunCat table:",
            cleaned_df.shape
        )

# Pipeline complete

For TOPMed, the generated feature tables are passed to the TrunCat model
workflow.

For gnomAD, ClinVar, and GREGoR, the corresponding feature tables can be
processed using the same annotation definitions and aligned to the TrunCat
predictor set for downstream prediction.

See the repository README files for dataset-specific provenance and external
reference-resource details.